# Phân Tích Cảm Xúc Tiếng Việt: BiLSTM + Attention vs. PhoBERT Fine-tune

**Đề tài**: Nghiên cứu & so sánh mô hình hồi quy hai chiều có cơ chế Attention tự huấn luyện với mô hình tiền huấn luyện PhoBERT trên ngữ liệu phản hồi của sinh viên (UIT-VSFC).
- **Tác giả**: Nguyễn Xuân Tú (`XuanTu138960`)
- **Bộ dữ liệu**: UIT-VSFC (16,000+ câu phản hồi của sinh viên từ nhóm UIT-NLP)
- **Papers tham khảo**: PhoBERT (EMNLP 2020), ViSoBERT (EMNLP 2023), ViDeBERTa (EACL 2023)

## 1. Cài Đặt Thư Viện & Thiết Lập Môi Trường

In [ ]:
# Cài đặt các thư viện phụ thuộc (nếu chạy trên Google Colab / Kaggle)
# !pip install torch transformers scikit-learn matplotlib seaborn pandas pyvi sentencepiece protobuf

In [ ]:
import os
import sys
import time
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập thiết bị tính toán
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Tải & Khám Phá Dữ Liệu UIT-VSFC

In [ ]:
data_dir = '../data'
df_train = pd.read_csv(os.path.join(data_dir, 'UIT-VSFC_train.csv')).dropna(subset=['text', 'label'])
df_valid = pd.read_csv(os.path.join(data_dir, 'UIT-VSFC_valid.csv')).dropna(subset=['text', 'label'])
df_test = pd.read_csv(os.path.join(data_dir, 'UIT-VSFC_test.csv')).dropna(subset=['text', 'label'])

print(f'Train set: {len(df_train)} rows')
print(f'Valid set: {len(df_valid)} rows')
print(f'Test set : {len(df_test)} rows')

print('\nPhân bố nhãn tập huấn luyện:')
print(df_train['label'].value_counts())

## 3. Tiền Xử Lý Tiếng Việt (Unicode, Teencode, Tách Từ PyVi)

In [ ]:
sys.path.append('..')
from src.preprocessor import VietnameseTextPreprocessor

preprocessor = VietnameseTextPreprocessor(segment=True)
samples = [
    'thầy dạy rất nhiệttt tìnhhhhh, e cảm ơn thầy nhiềuuuu <3',
    'slide bài giảng khó hiểu wa, ko có ví dụ thực tế j cả!!',
    'môn này rất ok, gv giải đáp thắc mắc cho sv rất nhanh.'
]

for s in samples:
    print(f'Gốc   : {s}')
    print(f'Xử lý : {preprocessor.preprocess(s)}\n')

## 4. Huấn Luyện BiLSTM + Attention

In [ ]:
# Chạy huấn luyện mô hình BiLSTM + Additive Attention
from src.train_bilstm import run_train_bilstm

# summary_bilstm = run_train_bilstm(data_dir=data_dir, output_dir='../outputs', epochs=6)

## 5. Fine-tuning PhoBERT (vinai/phobert-base-v2)

In [ ]:
# Chạy tinh chỉnh mô hình Transformer PhoBERT
from src.train_phobert import run_train_phobert

# summary_phobert = run_train_phobert(data_dir=data_dir, output_dir='../outputs', epochs=2, max_train_samples=2500)

## 6. Trực Quan Hóa Trọng Số Attention

In [ ]:
from src.visualize_attention import run_attention_demo
run_attention_demo(project_dir='..')

from IPython.display import Image
Image(filename='../outputs/figures/attention_visualization.png')

## 7. Đánh Giá & So Sánh Toàn Diện (Accuracy, Macro-F1, Confusion Matrix, Tài Nguyên)

In [ ]:
from src.evaluate_and_compare import run_evaluation_and_comparison
metrics_df = run_evaluation_and_comparison(project_dir='..')

Image(filename='../outputs/figures/model_comparison_chart.png')

## 8. Kết Luận & Định Hướng Sử Dụng

- **PhoBERT** phù hợp cho các bài toán đòi hỏi độ chính xác cao nhất (91.38%), ngữ cảnh mỉa mai, few-shot và chạy trên server GPU.
- **BiLSTM + Attention** tối ưu vượt trội về độ nhẹ (11 MB), độ trễ suy luận (1.79 ms/câu - nhanh gấp 24.4 lần), khả năng giải thích Attention trực tiếp và triển khai trên thiết bị biên/CPU.